<a href="https://colab.research.google.com/github/ArishaRamzan-dev/arisha-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArishaRamzan-dev/arisha-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# ===== SETUP — load data (same as w04) =====
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/ArishaRamzan-dev/arisha-flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# ===== LABEL: observed decline, not a future prediction =====
df['underperforming'] = (df['trend_pct'] < 0).astype(int)

# ===== FEATURES: exclude anything that defines or leaks into the label =====
leak_cols = ['trend_pct', 'trend_direction', 'impressions_last_30d', 'clicks_last_30d',
             'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
             'content_id', 'client_id', 'underperforming', 'action_score', 'reason_code', 'action_label',
             'staleness_bucket', 'position_bucket']
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in leak_cols]

print("Label balance:", df['underperforming'].value_counts(normalize=True).round(2).to_dict())
print("Feature count:", len(feature_cols))

Label balance: {1: 0.66, 0: 0.34}
Feature count: 23


In [ ]:
# Recreate Week-4 baseline rule so we can compare against it here
median_ctr = df['ctr'].median()

def score_row(row):
    score = 0
    if row['days_since_last_update'] > 180:
        score += 50
    if row['avg_position'] <= 10 and row['ctr'] < median_ctr:
        score += 30
    return score

df['action_score'] = df.apply(score_row, axis=1)
print("Baseline action_score recreated. Sample:", df['action_score'].value_counts().to_dict())

Baseline action_score recreated. Sample: {0: 23580, 30: 6246, 50: 99, 80: 75}


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm using Random Forest as the primary model, Logistic Regression as a linear check. The label
(underperforming, from trend_pct) likely has non-linear interactions with features like
avg_position and content_age_days — a mid-position, aging page might decline for different
reasons than a top-position new page. Random Forest captures that without manual interaction
terms, and gives permutation importance for the error analysis in Section 4.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
print("Random Forest (primary) vs Logistic Regression (linear check)")

Random Forest (primary) vs Logistic Regression (linear check)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client_id, same as the baseline, so no client's pages appear in both train and test
— this avoids the model learning client-specific quirks and inflating its score.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

X_train, X_test = df.loc[train_idx, feature_cols].fillna(0), df.loc[test_idx, feature_cols].fillna(0)
y_train, y_test = df.loc[train_idx, 'underperforming'], df.loc[test_idx, 'underperforming']

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

Train: 23837 rows | Test: 6163 rows


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Compared using Precision@50 — same idea as the baseline's top-20 review, extended to top-50 —
since the real use case is prioritizing a review queue, not classifying every row.

In [ ]:
def precision_at_k(y_true, y_scores, k=50):
    top_k_idx = np.argsort(y_scores)[-k:]
    return y_true.iloc[top_k_idx].mean()

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced").fit(X_train, y_train)

baseline_scores = df.loc[test_idx, 'action_score'].values  # your Week-4 rule, same test rows

results = pd.DataFrame({
    "Method": ["Week-4 Baseline (hand rule)", "Logistic Regression", "Random Forest"],
    "Precision@50": [
        precision_at_k(y_test, baseline_scores, 50),
        precision_at_k(y_test, log_reg.predict_proba(X_test)[:, 1], 50),
        precision_at_k(y_test, rf.predict_proba(X_test)[:, 1], 50),
    ]
})
print(results.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                     Method  Precision@50
Week-4 Baseline (hand rule)          0.60
        Logistic Regression          0.88
              Random Forest          0.76


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model leans almost entirely on one feature: days_with_impressions accounts for the large
majority of predictive signal (importance ≈0.074), about 4x the next feature (impressions_90d
≈0.017), with most other features — ctr, scroll_events_90d, engagement_rate — sitting near zero
or slightly negative, meaning they're mostly noise for this particular label. In practice the
model has learned something close to "how many days this page had any impressions at all,"
rather than a rich multi-signal picture.

The model misclassifies 39.4% of the test set (2,431 of 6,163 rows) — a meaningful error rate,
not a clean win. Notably, Logistic Regression (Precision@50 = 0.88) actually outperformed
Random Forest (0.76) here, contradicting my Section 1 hypothesis that non-linear interactions
would favor the more complex model. Both comfortably beat the Week-4 baseline (0.60), but this
result is a useful honest check: the relationship between top features and this label may be
simpler and more linear than expected, and the added complexity of Random Forest didn't buy
extra precision on this split. That's a directional, decision-support signal, not proof the
model has found a deep causal driver of page performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({"feature": feature_cols, "importance": perm.importances_mean}).sort_values("importance", ascending=False)
print(importance_df.head(10))

preds = rf.predict(X_test)
errors = X_test[y_test.values != preds]
print(f"\nMisclassified: {len(errors)} of {len(X_test)} ({len(errors)/len(X_test):.1%})")

                  feature  importance
13  days_with_impressions    0.074379
5         impressions_90d    0.016810
11        ai_sessions_90d    0.000081
22         ai_traffic_pct    0.000032
21            scroll_rate   -0.000308
10   engaged_sessions_90d   -0.000389
20        engagement_rate   -0.000454
0           search_volume   -0.000487
12      scroll_events_90d   -0.000681
18                    ctr   -0.000974

Misclassified: 2431 of 6163 (39.4%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.